In [1]:
!pip install pvlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 74.0 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import pvlib
from pvlib.location import Location
from tqdm import tqdm
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML # Colab 내에서 움짤을 바로 보기 위함

mpl.rcParams['animation.embed_limit'] = 50.0  # 단위는 MB, 50MB까지 허용하겠다는 뜻

np.random.seed(1045)
# ==========================================
# [Step 1] 기본 시스템 및 위치 설정
# ==========================================
LATITUDE = 37.5665
LONGITUDE = 126.9780
TZ = 'Asia/Seoul'
site_location = Location(LATITUDE, LONGITUDE, tz=TZ, name='Seoul')

START_HOUR = 5
END_HOUR = 19
NUM_MODULES = 10
WINDOW_SIZE = 60
NOCT_INSTALLED = 43.0

cec_modules = pvlib.pvsystem.retrieve_sam('cecmod')
MODULE_DATA = cec_modules['Hanwha_Q_Cells_Q_PEAK_DUO_L_G5_2_390']

# ==========================================
# [Step 2] 구름 이벤트 클래스 (최종 수정본)
# ==========================================
class CloudEvent:
    def __init__(self):
        self.active = False
        self.start_t = 0
        self.duration = 0
        self.intensity = 0
        self.velocity = 0.0
        self.width = 0.0
        self.event_type = 'moving'

    def trigger(self, t):
        self.active = True
        self.start_t = t
        self.intensity = np.random.uniform(0.3, 0.8)

        if np.random.rand() < 0.5:
            self.event_type = 'moving'
            self.velocity = np.random.uniform(0.01, 0.1)
            self.width = np.random.uniform(1.0, 4.0)
            # moving의 경우 duration을 사용하지 않고 '위치'로 종료 판단
        else:
            self.event_type = 'uniform'
            self.duration = np.random.randint(300, 1200) # uniform일 때만 duration 사용
            self.velocity = 0.0
            self.width = 0.0

    def get_shading_factors(self, t):
        if not self.active: return np.ones(NUM_MODULES)

        elapsed = t - self.start_t
        factors = np.ones(NUM_MODULES)

        if self.event_type == 'moving':
            # 1. 이동하는 구름
            # 구름의 중심 위치 계산 (시작점: 0번 패널 앞쪽(-width)에서 출발)
            center = (elapsed * self.velocity) - self.width

            # [핵심 수정] 구름의 꼬리까지 마지막 패널(NUM_MODULES)을 완전히 빠져나갔는지 확인 (2026/04/06)
            if center > (NUM_MODULES + self.width):
                self.active = False
                return np.ones(NUM_MODULES)

            # 구름이 아직 지나가는 중이라면 음영 계산
            for i in range(NUM_MODULES):
                dist = abs(i - center)
                if dist < self.width: # width 범위 내에 들어온 패널만 음영 적용
                    shading = self.intensity * np.exp(-(dist**2) / (2 * (self.width/2)**2))
                    factors[i] = 1 - shading

        elif self.event_type == 'uniform':
            # 2. 전체 패널 균일 흐려짐 (2026/04/06)
            # 설정된 duration(사인파 1주기)이 끝나면 종료
            if elapsed >= self.duration:
                self.active = False
                return np.ones(NUM_MODULES)

            shading = self.intensity * np.sin(np.pi * (elapsed / self.duration))
            factors *= (1 - shading)

        return factors

# ==========================================
# [Step 3] P-V 곡선 애니메이션 데이터 수집 및 렌더링
# ==========================================
def create_pv_animation(target_date, start_hour=12, duration_minutes=30, frame_skip=5):
    """
    frame_skip: 5로 설정하면 5초당 1프레임씩 애니메이션으로 만듭니다. (렌더링 속도 최적화)
    """
    print(f"\n[{target_date}] P-V 곡선 애니메이션 데이터 생성 시작...")

    # 1. 시뮬레이션 시간 범위 대폭 축소 (예: 12:00 부터 30분간)
    start_time = f'{target_date} {start_hour:02d}:00:00'
    times = pd.date_range(start=start_time, periods=duration_minutes*60, freq='s', tz=TZ)
    total_seconds = len(times)

    # 2. 일사량 및 온도 환경 세팅 (기존 로직 동일)
    clearsky = site_location.get_clearsky(times)
    base_g_array = clearsky['ghi'].values

    G_matrix = np.zeros((total_seconds, NUM_MODULES))
    cloud = CloudEvent()

    for t in range(total_seconds):
        base_g = max(base_g_array[t], 20.0)
        # 구름 발생 확률 (짧은 시간 안에 구름을 확실히 보기 위해 확률을 조금 높임)
        if not cloud.active and np.random.rand() < (0.05):
            cloud.trigger(t)

        factors = cloud.get_shading_factors(t)
        G_matrix[t, :] = base_g * factors

    # 온도 계산 (기존 로직 동일)
    T_matrix = np.zeros((total_seconds, NUM_MODULES))
    month = times[0].month
    temp_air_base = 15.0 - 15.0 * np.cos(np.pi * (month - 1) / 6.0)
    hours = times.hour + (times.minute / 60.0) + (times.second / 3600.0)
    daily_amplitude = 5.0
    temp_air_hourly = temp_air_base + daily_amplitude * np.sin(np.pi * (hours - 7.0) / 12.0)
    temp_air_series = pd.Series(temp_air_hourly, index=times)
    wind_speed_series = pd.Series(1.0, index=times)

    for i in range(NUM_MODULES):
        poa_series = pd.Series(G_matrix[:, i], index=times)
        t_cell = pvlib.temperature.fuentes(
            poa_global=poa_series, temp_air=temp_air_series,
            wind_speed=wind_speed_series, noct_installed=NOCT_INSTALLED
        )
        T_matrix[:, i] = t_cell.values

    # ==========================================
    # 물리 엔진 계산 및 애니메이션 프레임 저장
    # (LSTM 저장은 모두 제거, 시각화 데이터만 Append)
    # ==========================================
    v_now = MODULE_DATA['V_oc_ref'] * NUM_MODULES * 0.8
    v_old, p_old = v_now, 0
    direction = 1

    # 애니메이션을 위해 저장할 데이터 리스트
    frames_v_curve = []
    frames_p_curve = []
    frames_gmpp = []
    frames_mppt = []
    frames_time = []

    for t in tqdm(range(total_seconds), desc="물리엔진 및 MPPT 연산 중"):
        current_gs = G_matrix[t]
        current_ts = T_matrix[t]

        i_array = np.linspace(0, MODULE_DATA['I_sc_ref'] + 1, 300)
        v_total = np.zeros_like(i_array)

        for g, temp in zip(current_gs, current_ts):
            if g < 10: g = 10
            IL, Io, Rs, Rsh, nNsVth = pvlib.pvsystem.calcparams_desoto(
                effective_irradiance=g, temp_cell=temp,
                alpha_sc=MODULE_DATA['alpha_sc'], a_ref=MODULE_DATA['a_ref'],
                I_L_ref=MODULE_DATA['I_L_ref'], I_o_ref=MODULE_DATA['I_o_ref'],
                R_sh_ref=MODULE_DATA['R_sh_ref'], R_s=MODULE_DATA['R_s'],
                EgRef=1.121, dEgdT=-0.0002677
            )
            v_mod = pvlib.pvsystem.v_from_i(
                resistance_shunt=Rsh, resistance_series=Rs, nNsVth=nNsVth,
                saturation_current=Io, photocurrent=IL, current=i_array
            )
            v_total += np.maximum(v_mod, -0.7)

        p_total = v_total * i_array
        valid = p_total > 0
        p_clean, v_clean, i_clean = p_total[valid], v_total[valid], i_array[valid]
        gmpp_idx = np.argmax(p_clean)
        true_v, true_p = v_clean[gmpp_idx], p_clean[gmpp_idx]

        # 오름차순 정렬 후 보간
        idx_sort = np.argsort(v_clean)
        v_clean_sorted = v_clean[idx_sort]
        i_clean_sorted = i_clean[idx_sort]

        i_measured = np.interp(v_now, v_clean_sorted, i_clean_sorted)
        p_measured = v_now * i_measured

        # P&O 갱신
        dv = v_now - v_old
        if p_measured < p_old:
            direction *= -1

        v_old, p_old = v_now, p_measured
        v_now = np.clip(v_now + direction * 2.0, 10, v_clean_sorted[-1] if len(v_clean)>0 else 400)

        # 시스템 보호용 Jump (LSTM 코드는 삭제했지만 물리적 궤도를 위해 남김)
        if true_p > 0:
            ratio = p_measured / true_p
            if ratio <= 0.95:
                v_now = true_v

        # --- [추가] 프레임 데이터 저장 (frame_skip 주기마다) ---
        if t % frame_skip == 0:
            frames_v_curve.append(v_clean)
            frames_p_curve.append(p_clean)
            frames_gmpp.append((true_v, true_p))
            frames_mppt.append((v_old, p_measured))
            frames_time.append(str(times[t].time()))

    # ==========================================
    # 애니메이션(움짤) 그리기 파트
    # ==========================================
    print("\n데이터 연산 완료. 애니메이션 렌더링을 시작합니다...")
    fig, ax = plt.subplots(figsize=(10, 6))

    # 플롯 요소들 초기화
    line_curve, = ax.plot([], [], 'b-', lw=2, label='Real-time P-V Curve')
    point_gmpp, = ax.plot([], [], 'r*', markersize=15, label='True GMPP')
    point_mppt, = ax.plot([], [], 'go', markersize=8, label='P&O Tracker')
    time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=12, fontweight='bold')

    ax.set_xlim(0, 450)
    ax.set_ylim(0, 4000)
    ax.set_xlabel('Voltage (V)')
    ax.set_ylabel('Power (W)')
    ax.set_title('Dynamic P-V Curve & P&O MPPT Tracking')
    ax.legend(loc='upper right')
    ax.grid(True)

    def init():
        line_curve.set_data([], [])
        point_gmpp.set_data([], [])
        point_mppt.set_data([], [])
        time_text.set_text('')
        return line_curve, point_gmpp, point_mppt, time_text

    def update(frame):
        # 매 프레임마다 곡선과 점의 위치를 갱신
        line_curve.set_data(frames_v_curve[frame], frames_p_curve[frame])

        g_v, g_p = frames_gmpp[frame]
        point_gmpp.set_data([g_v], [g_p]) # 리스트 형태로 전달

        m_v, m_p = frames_mppt[frame]
        point_mppt.set_data([m_v], [m_p]) # 리스트 형태로 전달

        time_text.set_text(f"Time: {frames_time[frame]}")
        return line_curve, point_gmpp, point_mppt, time_text

    ani = FuncAnimation(fig, update, frames=len(frames_time),
                        init_func=init, blit=True, interval=50) # interval: 프레임당 밀리초(속도 조절)

    plt.close(fig) # 불필요한 빈 정적 그래프 출력 방지
    return ani

# ==========================================
# [Step 4] 실행부 (애니메이션 구동)
# ==========================================
if __name__ == "__main__":
    # 2026년 4월 15일, 오후 12시부터 30분간의 데이터를 5초 단위로 건너뛰며 움짤 생성
    ani = create_pv_animation("2026-07-15", start_hour=14, duration_minutes=60, frame_skip=5)

    # 1. Colab 환경에서 브라우저로 바로 보기 (HTML 비디오 태그 변환)
    display(HTML(ani.to_jshtml()))

    # 2. 파일로 저장하고 싶다면 아래 주석을 해제하세요. (저장 시 시간 좀 걸림)
    # save_path = '/content/drive/MyDrive/pv_mppt_animation.gif'
    # print("GIF 파일로 저장 중입니다...")
    # ani.save(save_path, writer='pillow', fps=20)
    # print(f"저장 완료: {save_path}")

Output hidden; open in https://colab.research.google.com to view.

In [3]:
# 2. 파일로 저장하고 싶다면 아래 주석을 해제하세요. (저장 시 시간 좀 걸림)
save_path = 'pv_mppt_animation.gif'
print("GIF 파일로 저장 중입니다...")
ani.save(save_path, writer='pillow', fps=20)
print(f"저장 완료: {save_path}")

GIF 파일로 저장 중입니다...
저장 완료: pv_mppt_animation.gif
